# Advanced Company Brochure Maker

**LLM Engineering, Assignment 1.** A source-based information kit about two companies in the same field:
four Hebrew brochures (potential customers and job candidates for each company), a comparison table,
and a baseline brochure for the prompt experiment.

This notebook extends the class notebook `day5.ipynb` (collect links → choose with an LLM → collect content →
write a brochure → stream it). The logic lives in the `brochure/` package; this notebook runs it step by step,
shows the results, and holds the manual review.

![Architecture](docs/architecture.svg)

**How to run:** fill in the two companies in section 2, then run the cells in order.
A full run makes about 8 model calls: 2 link selections, 4 brochures, 1 comparison and 1 baseline.
Section 10 re-runs all checks from the saved files of a run, with no model calls.

## 1. Setup

In [1]:
from IPython.display import HTML, Markdown, display

from brochure import audit, config
from brochure.checks import check_saved_output, format_checks
from brochure.config import CONTEXT_CHAR_BUDGET, MODEL, Company, check_api_key, get_companies
from brochure.context import build_context, format_report
from brochure.generate import generate_brochure, generate_comparison, rtl_markdown
from brochure.links import build_candidates, save_candidates, select_links
from brochure.runlog import RunLog
from brochure.sources import (add_replacement_source, collect_sources, fetch_home, load_sources,
                              save_sources, summarize_sources)

check_api_key()
print(f"Model: {MODEL} | context budget: {CONTEXT_CHAR_BUDGET:,} characters of source text per call")

API key found and looks like a Claude API key.
Model: claude-opus-5-5 | context budget: 12,000 characters of source text per call


## 2. The two companies

Choose two companies **in the same field** whose sites can be read by the class scraper
(`scraper.py`). For each company we use the home page and up to 3 more pages, with no recursive crawling.

- `company_id`: a short slug used in file names (e.g. `"monday"`)
- `name`: the display name
- `url`: the home page

In [ ]:
COMPANY_A = Company(company_id="TODO_a", name="TODO", url="https://TODO", letter="A")
COMPANY_B = Company(company_id="TODO_b", name="TODO", url="https://TODO", letter="B")
FIELD = "TODO: the field both companies work in"

config.COMPANIES[:] = [COMPANY_A, COMPANY_B]
companies = get_companies()
assert all("TODO" not in c.company_id + c.name + c.url for c in companies), "Fill in both companies first."
for c in companies:
    print(f"{c.letter}: {c.name} ({c.url})  id={c.company_id}")
print("Field:", FIELD)

**Site choice notes (fill in).** Why these two companies, and any site that was tried and replaced:

| Company | Site | Readable by the scraper? | Notes (e.g. replaced because the site is JavaScript-only) |
|---|---|---|---|
| A | | | |
| B | | | |

## 3. Start a run

Everything from this run is saved under `runs/<run_id>/`: candidates, link selections, sources,
exact contexts and prompts, outputs with their metadata and checks, and the log (`log.jsonl`).

In [ ]:
run = RunLog()
print("Run ID:", run.run_id)
print("Folder:", run.dir)

## 4. Links and sources (per company)

**4a. Candidate links, built in code.** The home page is fetched once. Its links are resolved, `#fragments` are
removed, `mailto:` / `tel:` / `javascript:` and links to other sites are dropped (`www.` and the bare domain
count as the same site), the home page, privacy / terms pages and file links are dropped, and duplicates are
removed. The resulting list is saved, since it is exactly what the model receives.

In [ ]:
homes, candidates, selections, collections = {}, {}, {}, {}

for co in companies:
    home = fetch_home(co, run=run)
    homes[co.company_id] = home
    print(f"{co.name}: HTTP {home['http_status']}, final URL {home['final_url']}, "
          f"{len(home['text']):,} chars of text, error: {home['error']}")
    assert home["error"] is None, f"The home page of {co.name} could not be read - choose another site."

    cands = build_candidates(co, home)
    save_candidates(cands, run.dir / "candidates")
    candidates[co.company_id] = cands
    print(f"  {cands['raw_link_count']} links on the page -> {len(cands['candidates'])} candidates "
          f"({len(cands['dropped'])} dropped, reasons saved)\n")

**4b. Link selection by the model (2 calls), validated in code.** The model picks up to 3 pages, each with a
`type`, `url` and `reason`. The answer is checked before anything is fetched. If anything is wrong, the whole
answer is rejected, its raw output and the reasons are saved, and nothing is fetched from it.
Re-running this cell retries only the rejected companies, and each retry is logged as a new attempt.

In [ ]:
for co in companies:
    if selections.get(co.company_id, {}).get("status") == "accepted":
        continue
    sel = select_links(co, candidates[co.company_id], run=run)
    selections[co.company_id] = sel
    print(f"{co.name}: {sel['status']} (attempt {sel['attempt']}, {sel['duration_seconds']}s, tokens {sel['usage']})")
    for link in sel["links"]:
        print(f"  - [{link['type']}] {link['normalized_url']}\n      reason: {link['reason']}")
    for reason in sel["rejection_reasons"]:
        print("  ! rejected:", reason)
    print()

not_accepted = [co.name for co in companies if selections[co.company_id]["status"] != "accepted"]
if not_accepted:
    raise RuntimeError(f"Link selection was not accepted for {not_accepted}: nothing is fetched from it. "
                       "The raw output and reasons are in selections/. Re-run this cell to try again.")

**4c. Source collection.** The home page becomes A1 / B1 and the selected pages follow in the model's order
(A2–A4 / B2–B4). Each source is `ok`, `empty` or `failed`. Empty and failed sources stay in the record so
the gaps are visible, but they are never used as evidence.

In [ ]:
for co in companies:
    col = collect_sources(co, homes[co.company_id], selections[co.company_id], run=run)
    save_sources(col, run.dir / "sources")
    collections[co.company_id] = col
    print(summarize_sources(col), "\n")

**If a company has too little usable information**, add a replacement page from its saved candidate list.
The replacement takes a free ID within the page limit, and the reason is recorded. Uncomment only if needed:

In [ ]:
# co = COMPANY_A
# add_replacement_source(collections[co.company_id], candidates[co.company_id],
#                        "https://...", "why this page is needed", run=run)
# save_sources(collections[co.company_id], run.dir / "sources")
# print(summarize_sources(collections[co.company_id]))

## 5. Contexts

All outputs are built from the **saved** sources, and nothing is fetched again. Only `ok` sources are used, within
a budget of source-text characters (`CONTEXT_CHAR_BUDGET` in `brochure/config.py`). The budget is split fairly,
so a long page is shortened instead of pushing the others out. Instructions, headers and source IDs are outside
the budget and are never cut. For the comparison, both companies share one budget.

In [ ]:
saved = {co.company_id: load_sources(run.dir / "sources" / f"{co.company_id}.json") for co in companies}

contexts = {co.company_id: build_context([saved[co.company_id]]) for co in companies}
contexts["comparison"] = build_context([saved[COMPANY_A.company_id], saved[COMPANY_B.company_id]])

for name, ctx in contexts.items():
    print(f"[{name}]")
    print(format_report(ctx), "\n")

## 6. Brochures (4 calls)

The same company facts serve two different readers. Each audience gets its own structure and emphasis.
Each brochure is checked in code right after it is generated: cited IDs, source list, word count (200–300 with `split`)
and required headings. The **first brochure is streamed**.

### 6.1 Company A, potential customers (streamed)

In [ ]:
results = {}
results["A_customers"] = generate_brochure(COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, stream=True)

### 6.2 Company A, job candidates

In [ ]:
results["A_candidates"] = generate_brochure(COMPANY_A, "candidates", contexts[COMPANY_A.company_id], run)

### 6.3 Company B, potential customers

In [ ]:
results["B_customers"] = generate_brochure(COMPANY_B, "customers", contexts[COMPANY_B.company_id], run)

### 6.4 Company B, job candidates

In [ ]:
results["B_candidates"] = generate_brochure(COMPANY_B, "candidates", contexts[COMPANY_B.company_id], run)

## 7. Comparison (1 call)

Four dimensions fixed before generation (`COMPARISON_DIMENSIONS` in `brochure/config.py`): products and services,
target audiences, pricing or trial, and working at the company. It is based on the companies' own publications, so it
is not independent market research.

In [ ]:
results["comparison"] = generate_comparison(COMPANY_A, COMPANY_B, contexts["comparison"], run)

## 8. Baseline brochure for the prompt experiment (1 call)

Same company, audience, model, settings and **exact same context** as brochure 6.1, with the same language, length and
source / citation rules. Only the prompt wording and the requested structure change (the class notebook's simple prompt).

In [ ]:
results["A_customers_baseline"] = generate_brochure(
    COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, variant="baseline")

## 9. Run summary

In [ ]:
rows = ["| Output | Run ID | Status | Words | Checks |", "|---|---|---|---|---|"]
for r in results.values():
    words = r["word_count"] if r["word_count"] is not None else "-"
    checks = "passed" if r["checks"]["passed"] else "; ".join(r["checks"]["problems"])
    rows.append(f"| {r['name']} | {r['run_id']} | {r['status']} | {words} | {checks} |")
display(Markdown("\n".join(rows)))

print(run.format_summary())
run.save_summary()

If a brochure fails a check (for example, the word count), keep the original output and either document it or
regenerate it. A regeneration is logged as attempt 2, saved as `<name>_attempt2`, and overwrites nothing.

## 10. Replay from saved files (no model calls)

Re-runs the checks on every saved output, and rebuilds every context from the saved sources to show it matches the
context that was actually sent. Set `REPLAY_RUN_ID` to any earlier run ID to re-check that run without calling the model.

In [ ]:
REPLAY_RUN_ID = run.run_id          # or an earlier run, e.g. "20261127-120000-abcd"
replay = RunLog(REPLAY_RUN_ID)

import json
saved_sources = {p.stem: load_sources(p) for p in (replay.dir / "sources").glob("*.json")}
for meta_path in sorted((replay.dir / "outputs").glob("*.json")):
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    name = meta_path.stem
    print(f"{name}: {format_checks(check_saved_output(replay.dir, name))}")

    ids = meta["company_id"].split("+")              # "a" for a brochure, "a+b" for the comparison
    rebuilt = build_context([saved_sources[i] for i in ids])
    sent = (replay.dir / "contexts" / f"{name}.txt").read_text(encoding="utf-8")
    print(f"   context rebuilt from saved sources == context sent: {rebuilt['text'] == sent}")

## 11. Manual claim audit (8 claims)

**Sampling rule (fixed in advance):** in each of the 4 improved brochures, take the **first two factual claims** in
reading order. A factual claim is a sentence or bullet in the body with at least one citation that is not a
"לא נמצא במקורות שנמסרו למודל" statement. Headings, the source list and table rows are not claims.
If a brochure has fewer than two, that is recorded as a gap.

The cell below applies the rule to the saved brochures and shows lines from the cited sources that share words with each
claim. That is only a reading aid: the verdict comes from reading the source. A cited ID existing is **not** proof that the
source supports the claim.

In [ ]:
AUDIT_NAMES = [f"{co.company_id}_{aud}" for co in companies for aud in ("customers", "candidates")]
claims = audit.sample_claims(replay.dir, AUDIT_NAMES, per_brochure=2)
print(audit.format_claims(claims))

audit_path = replay.dir / "audit" / "claim_audit.csv"
if not audit_path.exists():
    audit.write_audit_template(claims, audit_path)
print("Fill in evidence / verdict / note in:", audit_path)
print("Verdicts:", " | ".join(audit.VERDICTS))

**Audit results (fill in):**

| # | Brochure | Claim (as written) | Source ID | URL | Evidence quote from the source | Verdict | Correction or explanation |
|---|---|---|---|---|---|---|---|
| 1 | | | | | | | |
| 2 | | | | | | | |
| 3 | | | | | | | |
| 4 | | | | | | | |
| 5 | | | | | | | |
| 6 | | | | | | | |
| 7 | | | | | | | |
| 8 | | | | | | | |

The original outputs stay unchanged in `outputs/`. Any correction is written here.

## 12. Prompt experiment: baseline vs improved

**Criteria, fixed before reading the outputs:**
1. **Faithfulness:** every claim is supported by the cited source, with nothing invented and marketing claims attributed.
2. **Audience fit:** structure, order and explanations serve a potential customer.
3. **Information coverage:** problem, offering, uses / audiences, and open questions.
4. **Clarity:** easy to scan, with clear headings and no filler.

The same model, settings, context, audience, language, length and source rules were used for both. Only the
prompt wording and the requested structure differ. This is one small comparison, not a general ranking of prompts or models.

In [ ]:
improved, baseline = results["A_customers"], results["A_customers_baseline"]
for label, r in (("Improved", improved), ("Baseline", baseline)):
    print(f"{label}: {r['word_count']} words, cited {r['checks']['cited_ids']}, "
          f"'not found' x{r['checks']['missing_info_mentions']}, checks {'passed' if r['checks']['passed'] else r['checks']['problems']}")

def column(title, text):
    return (f'<div style="flex:1; min-width:300px; border:1px solid #ccc; border-radius:8px; padding:8px 16px">'
            f'<h3 dir="ltr">{title}</h3>\n\n{rtl_markdown(text).data}\n\n</div>')

display(Markdown('<div style="display:flex; gap:16px; flex-wrap:wrap">\n\n'
                 + column("Improved", improved["text"]) + column("Baseline", baseline["text"]) + "\n\n</div>"))

**Findings (fill in, with concrete examples quoted from both versions):**

| Criterion | Improved | Baseline | Example |
|---|---|---|---|
| Faithfulness | | | |
| Audience fit | | | |
| Information coverage | | | |
| Clarity | | | |

Where the improved prompt did **not** help, say so too.

## 13. Discussion (to complete)

- **Code vs model:** what the code does (links, validation, fetching, budget, checks, logging) and what the model does
  (choosing pages, writing).
- **Context management:** the character budget, fair split, what was shortened or left out in this run.
- **Validation points:** link selection, source status, citations / source list / word count, and the manual audit.
- **Changing the prompt vs changing the source material:** how each can change the output.
- **Workflow vs agent:** this is a fixed workflow. What would be needed for an agent that chooses tools at run time.